In [ ]:
%pip install kagglehub
import kagglehub
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib

In [ ]:
path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
print("Path to dataset files:", path)

In [ ]:
# Excel File Download Path
%pip install openpyxl

excel_files = [f for f in os.listdir(path) if f.endswith('.xlsx')]
filepath = os.path.join(path, excel_files[0])
print("Reading file:", filepath)

df = pd.read_excel(filepath, engine='openpyxl')

raw_dir = os.path.join('..', 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

raw_path = os.path.join(raw_dir, 'raw_telco_churn.csv')
df.to_csv(raw_path, index=False)
print(f"Raw dataset exported as '{raw_path}'")

print("Data shape (rows, columns):", df.shape)
df.head()

excel_files = [f for f in os.listdir(path) if f.endswith('.xlsx')]
filepath = os.path.join(path, excel_files[0])
print("Reading file:", filepath)

df = pd.read_excel(filepath)
raw_dir = os.path.join('..', 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

raw_path = os.path.join(raw_dir, 'raw_telco_churn.csv')
df.to_csv(raw_path, index=False)
print(f"Raw dataset exported as '{raw_path}'")

print("Data shape (rows, columns):", df.shape)

df.head()

In [ ]:
print("=== DATAFRAME INFO ===")
df.info()

print("\n=== CHURN DISTRIBUTION ===")
print(df['Churn Value'].value_counts())
print("\n=== CHURN PERCENTAGE ===")
print(df['Churn Value'].value_counts(normalize=True) * 100)

In [ ]:
# Blank Strings to NaN to 0
blank_count = (df['Total Charges'] == ' ').sum()
print(f"Blank 'Total Charges' entries found: {blank_count}")

df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

df['Total Charges'] = df['Total Charges'].fillna(0)

print("Updated 'Total Charges' data type:", df['Total Charges'].dtype)

In [ ]:
# Columns that leak target information or act as raw identifiers
cols_to_drop = [ 'CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Churn Label', 'Churn Reason', 'Churn Score', 'CLTV', 'Gender', 'Phone Service' ]
# Drop administrative & leakage columns
df_cleaned = df.drop(columns=cols_to_drop, errors='ignore')
print("Remaining features for modeling:", df_cleaned.shape[1])
print("Columns list:\n", df_cleaned.columns.tolist())

In [ ]:
# Create Tenure Cohorts
def get_tenure_cohort(months):
  if months <= 12:
    return 'Cohort_0_12'
  elif months <= 24:
    return 'Cohort_13_24'
  elif months <= 48:
    return 'Cohort_25_48'
  else:
    return 'Cohort_49_72'

df_cleaned['Tenure_Cohort'] = df_cleaned['Tenure Months'].apply(get_tenure_cohort)

# Total Subscribed Services
service_cols = [ 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies' ]

# No. of 'Yes' Service
df_cleaned['Total_Services'] = (df_cleaned[service_cols] == 'Yes').sum(axis=1)

# Monthly-to-Tenure Ratio
df_cleaned['Monthly_to_Tenure_Ratio'] = df_cleaned['Monthly Charges'] / (df_cleaned['Tenure Months'] + 1)
print("Engineered Features Added Successfully!")
print("New DataFrame shape:", df_cleaned.shape)
df_cleaned[['Tenure Months', 'Tenure_Cohort', 'Total_Services', 'Monthly_to_Tenure_Ratio']].head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

# 0. Separate Features (X) and Target (y), then Split
X = df_cleaned.drop(columns=['Churn Value'])
y = df_cleaned['Churn Value']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Define ALL Categorical Features (including remaining object columns)
nominal_features = [
    'Contract', 'Payment Method', 'Internet Service',
    'Multiple Lines', 'Online Security', 'Online Backup',
    'Device Protection', 'Tech Support', 'Streaming TV',
    'Streaming Movies', 'Tenure_Cohort'
]
binary_features = [
    'Senior Citizen', 'Partner', 'Dependents','Paperless Billing'
]
categorical_cols = nominal_features + binary_features

# 2. Define Continuous Numerical Features for Scaling
numeric_cols = [
    'Tenure Months', 'Monthly Charges', 'Total Charges',
    'Total_Services', 'Monthly_to_Tenure_Ratio'
]

# 3. Construct Unified Preprocessing Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)

# 4. Fit strictly on Train set & transform both Train and Test (Uppercase X!)
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Get clean feature names for the exported CSVs
feature_names = preprocessor.get_feature_names_out()
train_df = pd.DataFrame(X_train_processed, columns=feature_names)
train_df['Churn Value'] = y_train.values

test_df = pd.DataFrame(X_test_processed, columns=feature_names)
test_df['Churn Value'] = y_test.values

output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

# 5. Save the unified preprocessor artifact
preprocessor_path = os.path.join(output_dir, 'preprocessor.pkl')
joblib.dump(preprocessor, preprocessor_path)
print(f"\nSaved unified preprocessor to '{preprocessor_path}'!")

# 6. Export clean CSV files
train_path = os.path.join(output_dir, 'train_telco_churn.csv')
test_path = os.path.join(output_dir, 'test_telco_churn.csv')

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print("\n=== FINAL DELIVERABLES EXPORTED ===")
print(f"- {train_path}:", train_df.shape)
print(f"- {test_path}: ", test_df.shape)

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. Log-Transform the Ratio to Eliminate Extreme Skewness
df_cleaned['Log_Monthly_Tenure_Ratio'] = np.log1p(df_cleaned['Monthly_to_Tenure_Ratio'])

# 2. Multicollinearity Check (VIF Calculation with Intercept)
vif_cols = [
    'Tenure Months', 'Monthly Charges', 'Total Charges',
    'Total_Services', 'Log_Monthly_Tenure_Ratio'
]
X_vif = add_constant(df_cleaned[vif_cols].dropna())

vif_data = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
# Filter out the 'const' row for cleaner display
vif_data = vif_data[vif_data["Feature"] != "const"].reset_index(drop=True)

print("--- Variance Inflation Factor (VIF) Results ---")
print(vif_data.to_string(index=False))

# 3. Ablation Experiment (Leak-Free 5-Fold Stratified Cross-Validation)
base_categoricals = [
    'Contract', 'Payment Method', 'Internet Service',
    'Multiple Lines', 'Online Security', 'Online Backup',
    'Device Protection', 'Tech Support', 'Streaming TV',
    'Streaming Movies', 'Gender', 'Senior Citizen',
    'Partner', 'Dependents', 'Phone Service', 'Paperless Billing'
]
base_numerics = ['Tenure Months', 'Monthly Charges', 'Total Charges']

eng_categoricals = base_categoricals + ['Tenure_Cohort']
eng_numerics = base_numerics + ['Total_Services', 'Log_Monthly_Tenure_Ratio']

def evaluate_feature_set(X_subset, y_target, num_cols, cat_cols):
    prep = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])
    model_pipeline = Pipeline([
        ('preprocessor', prep),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    ])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(model_pipeline, X_subset, y_target, cv=cv, scoring=['roc_auc', 'recall'])
    return scores['test_roc_auc'].mean(), scores['test_recall'].mean()

y_target = df_cleaned['Churn Value']
X_raw = df_cleaned[base_numerics + base_categoricals]
X_eng = df_cleaned[eng_numerics + eng_categoricals]

raw_auc, raw_rec = evaluate_feature_set(X_raw, y_target, base_numerics, base_categoricals)
eng_auc, eng_rec = evaluate_feature_set(X_eng, y_target, eng_numerics, eng_categoricals)

print("\n--- Ablation Study Results (5-Fold CV Logistic Regression) ---")
print(f"Raw Features Only   -> ROC-AUC: {raw_auc:.4f} | Recall: {raw_rec:.4f}")
print(f"Raw + Eng. Features -> ROC-AUC: {eng_auc:.4f} | Recall: {eng_rec:.4f}")